# Practice 1. Reading & Converting Formats

**Course:** Data Storage and Analysis, KBTU
**Student:** Tomiris, ID 23B031507
**Objective:** select a dataset, analyze its structure and quality, prepare the data, and compare storage formats (CSV, JSON, Parquet, ORC, Avro).

## Task 1. Dataset Selection

**Dataset:** Titanic — Passenger Survival Data
**Source:** hosted as a CSV in the public `seaborn-data` repository (a standard open mirror of the classic Kaggle Titanic dataset — https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv), originally distributed by Kaggle (https://www.kaggle.com/c/titanic) and derived from the Encyclopedia Titanica passenger records.
**Author / organization:** Kaggle / seaborn (mwaskom), originating from the Encyclopedia Titanica project.

Each **row** represents one passenger aboard the RMS Titanic (1912 voyage), and each **column** describes an attribute of that passenger or their trip outcome.

In [1]:
import pandas as pd
import numpy as np
import time, os, json

pd.set_option('display.max_columns', 20)

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df_raw = pd.read_csv(URL)

print("Rows, Columns:", df_raw.shape)
df_raw.head()

Rows, Columns: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
print(df_raw.dtypes)

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object


**Column meanings:**

| Column | Meaning |
|---|---|
| survived | Survival outcome (0 = died, 1 = survived) |
| pclass | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) |
| sex | Passenger sex |
| age | Age in years (fractional for infants) |
| sibsp | # of siblings/spouses aboard |
| parch | # of parents/children aboard |
| fare | Ticket fare paid (£) |
| embarked | Port of embarkation code (S/C/Q) |
| class | Ticket class as text (redundant with pclass) |
| who | Man / woman / child classification |
| adult_male | Boolean flag for adult male |
| deck | Deck letter extracted from cabin number |
| embark_town | Full port name (redundant with embarked) |
| alive | Survival as yes/no (redundant with survived) |
| alone | Boolean flag: travelling alone |

**Dataset type:** structured (fixed tabular schema, one row per passenger, homogeneous column types).

## Task 2. Data-Quality Analysis

We check data types, missing values, duplicates, invalid values, inconsistent/redundant categories, and outliers.

In [2]:
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).query("missing_count > 0")

,missing_count,missing_pct
age,177,19.9
embarked,2,0.2
deck,688,77.2
embark_town,2,0.2


In [3]:
dupe_count = df_raw.duplicated().sum()
print("Fully duplicated rows:", dupe_count)
df_raw[df_raw.duplicated(keep=False)].sort_values(list(df_raw.columns)).head(6)

Fully duplicated rows: 107


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
64,0,1,male,NaN,0,0,27.7208,C,First,man,True,NaN,Cherbourg,no,True
295,0,1,male,NaN,0,0,27.7208,C,First,man,True,NaN,Cherbourg,no,True
144,0,2,male,18.0,0,0,11.5000,S,Second,man,True,NaN,Southampton,no,True
757,0,2,male,18.0,0,0,11.5000,S,Second,man,True,NaN,Southampton,no,True
658,0,2,male,23.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
733,0,2,male,23.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True


In [5]:
# Outlier check on numeric columns via IQR rule
for col in ["age", "fare"]:
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df_raw[col] < lower) | (df_raw[col] > upper)).sum()
    print(f"{col}: IQR bounds=({lower:.2f}, {upper:.2f}) -> {n_out} outliers "
          f"(min={df_raw[col].min()}, max={df_raw[col].max()})")

print("\nFree tickets (fare == 0):", (df_raw['fare'] == 0).sum())

age: IQR bounds=(-6.69, 64.81) -> 11 outliers (min=0.42, max=80.0)
fare: IQR bounds=(-26.72, 65.63) -> 116 outliers (min=0.0, max=512.3292)

Free tickets (fare == 0): 15


In [6]:
for col in ["sex", "embarked", "embark_town", "class", "who", "alive"]:
    print(col, "->", df_raw[col].unique())

sex -> <ArrowStringArray>
['male', 'female']
Length: 2, dtype: str
embarked -> <ArrowStringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str
embark_town -> <ArrowStringArray>
['Southampton', 'Cherbourg', 'Queenstown', nan]
Length: 4, dtype: str
class -> <ArrowStringArray>
['Third', 'First', 'Second']
Length: 3, dtype: str
who -> <ArrowStringArray>
['man', 'woman', 'child']
Length: 3, dtype: str
alive -> <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str


**Data-quality findings**

| Column | Problem | Example | Possible effect on analysis |
|---|---|---|---|
| age | 177 missing values (19.9%) | `NaN` for 177 passengers | Biases mean/median age; breaks age-based models unless imputed or the rows are dropped |
| deck | 688 missing values (77.2%) | `NaN` for most rows | Column is largely unusable as-is; needs to be dropped or treated as a separate "unknown" category |
| embarked / embark_town | 2 missing values each | `NaN` | Small impact, but the two missing rows must be handled consistently across both linked columns |
| (whole row) | 107 fully duplicated rows | identical `survived, pclass, sex, age, ...` combinations | No unique passenger ID exists in this extract, so duplicates cannot be confirmed as true repeated records; inflates counts/statistics if treated naively |
| fare | Outliers / invalid-looking values | `fare = 512.33` (max) and 15 rows with `fare = 0` | Skews mean fare and any fare-based model; £0 fares (crew/comp tickets) need a business-rule decision, not blind removal |
| sex / who / adult_male | Redundant/overlapping categorical encodings | `sex="male"`, `who="man"`, `adult_male=True` all encode similar info | Risk of double-counting the same signal in a model; inconsistent category granularity (who has 3 levels, sex has 2) |
| embarked / embark_town | Redundant dual encoding (code vs. full name) | `embarked="S"` vs `embark_town="Southampton"` | Same information stored twice in different formats — a data-consistency/normalization issue |
| survived / alive | Redundant dual encoding (numeric vs. text) | `survived=1` vs `alive="yes"` | Same information stored twice as different types (int vs string) |
| age | Non-integer values for infants | `age = 0.42` | Correct in context (a 5-month-old), but a naive "age must be integer" validation rule would flag it as invalid |

## Task 3. Data Formats — Preparing and Saving the Dataset

**Preparation applied before saving:**
1. Drop the 107 fully-duplicated rows (no unique passenger ID exists to verify them as distinct people, so we treat repeats conservatively).
2. Drop the `deck` column (77% missing — not usable without a dedicated imputation strategy).
3. Keep `age` missing values as-is (imputing age is a modeling decision outside the scope of format preparation).
4. Cast the two boolean columns and the categorical text columns to explicit `category`/`bool` dtypes so the schema is well-defined before export.

In [7]:
df = df_raw.drop_duplicates().drop(columns=["deck"]).reset_index(drop=True)

for c in ["sex", "embarked", "class", "who", "embark_town", "alive"]:
    df[c] = df[c].astype("category")
for c in ["adult_male", "alone"]:
    df[c] = df[c].astype("bool")

print("Prepared shape:", df.shape)
df.dtypes

Prepared shape: (784, 14)


survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
embarked       category
class          category
who            category
adult_male         bool
embark_town    category
alive          category
alone              bool
dtype: object

**Chosen formats:** **CSV** (primary/suitable format) and **Parquet** (alternative format).

*Why CSV as the primary format:* the dataset is small (~180 KB), flat, and fully tabular with no nested structures — CSV is human-readable, opens in Excel/any text editor, and is the universal exchange format expected for a submitted coursework dataset.

*Why Parquet as the alternative:* it is the natural "next step" for the same tabular data once volume or downstream analytics matter — columnar storage, built-in schema/dtype preservation (so `category`/`bool` types survive a round-trip, unlike CSV), and compression, which is useful if this dataset were scaled up.

In [8]:
os.makedirs("data", exist_ok=True)
df.to_csv("data/titanic_prepared.csv", index=False)
df.to_parquet("data/titanic_prepared.parquet", index=False, engine="pyarrow")
df.to_json("data/titanic_prepared.json", orient="records")  # for the format comparison in Task 4

for f in ["titanic_prepared.csv", "titanic_prepared.parquet", "titanic_prepared.json"]:
    print(f, "->", os.path.getsize(f"data/{f}"), "bytes")

titanic_prepared.csv -> 49585 bytes
titanic_prepared.parquet -> 14163 bytes
titanic_prepared.json -> 157331 bytes


## Task 4. Data Format Comparison

### 4.1 General comparison: CSV, JSON, Parquet, ORC, Avro

| Aspect | CSV | JSON | Parquet | ORC | Avro |
|---|---|---|---|---|---|
| Data representation | Flat text, comma-separated rows | Text, nested key/value objects | Binary, columnar | Binary, columnar | Binary, row-oriented |
| Storage structure | Row-based | Row-based (tree/nested) | Column-based, split into row groups | Column-based, split into stripes | Row-based, records serialized with schema |
| Schema support | None (types inferred on read) | None enforced (self-describing but no strict schema) | Strong — schema embedded in file footer | Strong — schema embedded, with built-in indexes/statistics | Strong — schema stored separately/embedded as JSON, schema evolution supported |
| Human readability | High (plain text) | High (plain text, nested) | None (binary) | None (binary) | None (binary) |
| Compression | None natively (external gzip possible) | None natively | Built-in, columnar (Snappy/GZIP) — very efficient | Built-in, columnar, often best-in-class | Built-in, moderate |
| Typical applications | Data exchange, spreadsheets, small/medium tabular data | Web APIs, config, semi-structured/nested data | Big-data analytics, data lakes (Spark, Hive, Presto) | Hadoop/Hive analytical workloads | Streaming pipelines (Kafka), schema evolution across services |
| Advantages | Simple, universal, human-readable, easy to edit | Flexible/nested, language-agnostic, widely used in APIs | Fast columnar reads, compact, schema-safe, great for analytics | Excellent compression, fast for Hive/Hadoop analytic scans | Compact, fast serialization, robust schema evolution |
| Limitations | No schema/types, no compression, slow on large data, no nesting | Verbose, no compression, slower to parse, no columnar reads | Not human-readable, needs libraries (pyarrow/fastparquet), poor for row-by-row access | Not human-readable, most tied to the Hadoop ecosystem | Not human-readable, less convenient for ad-hoc analytics/columnar scans |

### 4.2 Empirical comparison: CSV vs. Parquet (the two formats used above)

We measure file size, write time, read time, and whether rows/columns/dtypes/missing values survive the round trip.

In [9]:
results = []

# --- CSV ---
t0 = time.perf_counter()
df.to_csv("data/bench.csv", index=False)
csv_write = time.perf_counter() - t0

t0 = time.perf_counter()
df_csv = pd.read_csv("data/bench.csv")
csv_read = time.perf_counter() - t0

results.append({
    "format": "CSV",
    "size_bytes": os.path.getsize("data/bench.csv"),
    "write_time_s": round(csv_write, 5),
    "read_time_s": round(csv_read, 5),
    "rows_preserved": df_csv.shape[0] == df.shape[0],
    "cols_preserved": df_csv.shape[1] == df.shape[1],
    "dtypes_preserved": df_csv.dtypes.astype(str).equals(df.dtypes.astype(str)),
    "missing_preserved": df_csv.isna().sum().sum() == df.isna().sum().sum(),
})

# --- Parquet ---
t0 = time.perf_counter()
df.to_parquet("data/bench.parquet", index=False, engine="pyarrow")
pq_write = time.perf_counter() - t0

t0 = time.perf_counter()
df_pq = pd.read_parquet("data/bench.parquet", engine="pyarrow")
pq_read = time.perf_counter() - t0

results.append({
    "format": "Parquet",
    "size_bytes": os.path.getsize("data/bench.parquet"),
    "write_time_s": round(pq_write, 5),
    "read_time_s": round(pq_read, 5),
    "rows_preserved": df_pq.shape[0] == df.shape[0],
    "cols_preserved": df_pq.shape[1] == df.shape[1],
    "dtypes_preserved": df_pq.dtypes.astype(str).equals(df.dtypes.astype(str)),
    "missing_preserved": df_pq.isna().sum().sum() == df.isna().sum().sum(),
})

bench_df = pd.DataFrame(results)
bench_df

,format,size_bytes,write_time_s,read_time_s,rows_preserved,cols_preserved,dtypes_preserved,missing_preserved
0,CSV,49585,0.00402,0.00272,True,True,False,True
1,Parquet,14163,0.00251,0.01114,True,True,True,True


In [10]:
print(f"CSV dtypes after reload:\n{df_csv.dtypes}\n")
print(f"Parquet dtypes after reload:\n{df_pq.dtypes}")

CSV dtypes after reload:
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
embark_town        str
alive              str
alone             bool
dtype: object

Parquet dtypes after reload:
survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
embarked       category
class          category
who            category
adult_male         bool
embark_town    category
alive          category
alone              bool
dtype: object


**Observations (measured above):**
- Both formats preserve the number of **rows and columns**, and both preserve the **count of missing values**.
- **Dtypes are NOT preserved by CSV** — `category`/`bool` columns come back as plain `object`/string on reload (`dtypes_preserved = False`), because CSV stores no schema. **Parquet preserves dtypes exactly** (`dtypes_preserved = True`), since the schema is embedded in the file footer.
- **File size:** Parquet (≈14 KB) is roughly 3.5× smaller than CSV (≈50 KB) thanks to columnar layout and built-in compression.
- **Write time:** in this run, Parquet wrote slightly faster than CSV (≈0.0025s vs ≈0.0040s) — pyarrow's columnar writer is efficient even at this small scale.
- **Read time:** CSV read faster than Parquet (≈0.0027s vs ≈0.0111s) at this small size, since Parquet carries per-file overhead (footer/metadata parsing, columnar reconstruction) that only pays off on larger datasets. At hundreds of thousands+ rows, Parquet typically overtakes CSV on both read and write speed as well as size.

**Conclusion:** for this Titanic-sized, flat, tabular dataset, **CSV remains the most appropriate primary format** for the coursework submission — it is small, human-readable, and needs no special libraries to inspect. **Parquet is the recommended alternative** if the dataset were used in an analytical pipeline, thanks to its smaller size and full schema/dtype preservation.